# Qwen3 Embedding Domain Specific Fine-Tuning

Prerequisites:
- 기본적인 Python / pandas 사용
- embedding, cosine similarity, Top-k retrieval 개념
- Day-06 Lab 0에서 만든 `embedding_train.jsonl`, `embedding_eval.jsonl`
  - 스키마: `query`, `positive`, `hard_negative`, `task_instruction`, `source_doc`, `split`

Learning goals:
- `Qwen3-Embedding-0.6B`를 Unsloth로 도메인 특화 튜닝한다.
- hold-out query 30개 기준으로 튜닝 전후 `Top-3 hit rate`를 비교한다.
- query-side 영어 instruction prefix가 retrieval에 왜 중요한지 작은 실험으로 확인한다.
- retrieval 품질 개선과 generative QA 품질 개선이 다른 문제라는 점을 구분한다.
- 벡터 압축(uint8/binary)으로 메모리 절감 효과를 확인하고, 배포 시 trade-off를 이해한다.

## Outline

| Step | 내용 | 핵심 질문 |
|------|------|----------|
| 1 | 환경 설치 | GPU가 준비되었는가? |
| 2 | 데이터 계약 확인 + 전처리 | 학습 데이터는 어떤 형태여야 하는가? |
| 3 | Baseline retrieval 측정 | 튜닝 전 검색 품질은 어느 수준인가? |
| 4 | Instruction prefix 실험 | 영어 instruction이 한국어 검색에 도움이 되는가? |
| 5 | LoRA + MNRL 학습 | contrastive learning으로 embedding을 어떻게 조정하는가? |
| 6 | Before/After 비교 | 튜닝이 실제로 retrieval을 개선했는가? |
| 7 | 벡터 압축 (uint8/binary) | 품질을 유지하면서 메모리를 얼마나 줄일 수 있는가? |
| 8 | 결과 해석 + Exercise | 어떤 문제가 retrieval이고 어떤 문제가 generation인가? |

고정 제약:

| 항목 | 값 |
|------|-----|
| 기본 모델 | `Qwen3-Embedding-0.6B` |
| 비교 지표 | `Top-3 hit rate` |
| Loss | `MultipleNegativesRankingLoss` |
| `task_instruction` 언어 | 영어 |

## Official References

실습 중 헷갈릴 때 바로 들어가서 확인할 1차 문서들이다.

- Unsloth 공식 Qwen3-Embedding-0.6B notebook: https://github.com/unslothai/notebooks/blob/main/nb/Qwen3_Embedding_%280_6B%29.ipynb
- Unsloth notebooks 저장소: https://github.com/unslothai/notebooks
- Qwen3-Embedding 공식 저장소: https://github.com/QwenLM/Qwen3-Embedding
- Sentence Transformers loss 문서: https://www.sbert.net/docs/package_reference/sentence_transformer/losses.html

In [ ]:
from __future__ import annotations

import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)

SEED

## Step 1 - Environment Setup

이 실습은 CUDA GPU가 필요하다. `unsloth`로 모델 로딩과 LoRA 학습을 단순화하고, `transformers==4.56.2`, `trl==0.22.2`로 버전을 고정한다.

> OOM이 발생하면 `lab2_instructor_notes.md`의 "OOM 대응 우선순위"를 참고한다.

In [ ]:
%%capture
import os
import re

if "COLAB_" not in "".join(os.environ.keys()):
    %pip install -q unsloth
else:
    import torch

    v = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + {
        "2.10": "0.0.34",
        "2.9": "0.0.33.post1",
        "2.8": "0.0.32.post2",
    }.get(v, "0.0.34")
    %pip install -q sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    %pip install -q --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
%pip install -q transformers==4.56.2
%pip install -q --no-deps trl==0.22.2
%pip install -q sentence-transformers datasets pandas numpy
%pip install -q faiss-cpu

In [ ]:
# ---------- transformers 버전 검증 ----------
# Lab 1은 transformers==5.2.0, Lab 2는 transformers==4.56.2를 사용합니다.
# Lab 1을 같은 커널에서 실행했다면 반드시 커널을 재시작하세요.
import transformers

assert transformers.__version__.startswith("4.56"), (
    f"Expected transformers==4.56.x but got {transformers.__version__}. "
    "Lab 1을 같은 커널에서 실행했다면 커널을 재시작하세요."
)
print(f"transformers version: {transformers.__version__} ✓")

In [ ]:
import torch

BASE_MODEL_NAME = "unsloth/Qwen3-Embedding-0.6B"
EXTENSION_MODEL_NAME = "unsloth/Qwen3-Embedding-4B"  # optional extension track
MAX_SEQ_LENGTH = 512
USE_GRADIENT_CHECKPOINTING = False  # if OOM, change to "unsloth"

NOTEBOOK_DIR = Path(".")
RUNPOD_PERSISTENT_ROOT = Path("/workspace")
DATA_DIR = NOTEBOOK_DIR / "data"
TRAIN_PATH = DATA_DIR / "embedding_train.jsonl"
EVAL_PATH = DATA_DIR / "embedding_eval.jsonl"

RUN_NAME = "day06_lab2_qwen3_embedding"
ARTIFACT_ROOT = (
    RUNPOD_PERSISTENT_ROOT if RUNPOD_PERSISTENT_ROOT.exists() else NOTEBOOK_DIR
)
ARTIFACT_DIR = ARTIFACT_ROOT / "artifacts" / RUN_NAME
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

NUM_EVAL_QUERIES = 30
TOP_K_CLASS = 3
TOP_K_MRR = 10
TRAIN_EPOCHS = 1
LEARNING_RATE = 3e-5

# ---------- H100/A100 자동 감지: GPU 메모리에 따라 배치 사이즈 조정 ----------
assert torch.cuda.is_available(), (
    "GPU is required for this lab. Connect a CUDA GPU in RunPod first."
)

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

if max_memory > 40:
    # H100 80GB / A100 80GB: 큰 배치 = in-batch negative 증가 → 학습 효과 향상
    BATCH_SIZE = 128
    GRAD_ACC_STEPS = 1
    print(f"🔧 고용량 GPU 감지 ({gpu_stats.name}, {max_memory}GB)")
    print(f"   → BATCH_SIZE={BATCH_SIZE} (in-batch negatives: {BATCH_SIZE - 1}개)")
    print(f"   → Contrastive learning에서 배치가 클수록 학습 신호가 강해집니다")
elif max_memory > 20:
    # 24GB GPU: 기본 설정
    BATCH_SIZE = 32
    GRAD_ACC_STEPS = 2
    print(f"🔧 24GB급 GPU 감지 ({gpu_stats.name}, {max_memory}GB)")
    print(f"   → BATCH_SIZE={BATCH_SIZE} (in-batch negatives: {BATCH_SIZE - 1}개)")
else:
    # 소형 GPU
    BATCH_SIZE = 16
    GRAD_ACC_STEPS = 4
    print(f"⚠ 소형 GPU ({gpu_stats.name}, {max_memory}GB) — 보수적 설정 적용")

RECOMMENDED_SAMPLE_QUERIES = [
    "가용성 SLA는 어떻게 계산하나요?",
    "TTFE와 E2E를 함께 보는 이유는 무엇인가요?",
    "오류율을 정의할 때 분모를 명확히 해야 하는 이유는 무엇인가요?",
    "운영 루프에서 Step 5, 6, 8의 순서는 왜 중요한가요?",
    "트레이스 중심 개발이 코드 중심 디버깅과 어떻게 다른가요?",
]

{
    "gpu": gpu_stats.name,
    "max_memory_gb": max_memory,
    "reserved_memory_gb": start_gpu_memory,
    "batch_size": BATCH_SIZE,
    "grad_acc_steps": GRAD_ACC_STEPS,
    "effective_batch": BATCH_SIZE * GRAD_ACC_STEPS,
    "artifact_dir": str(ARTIFACT_DIR),
    "use_gradient_checkpointing": USE_GRADIENT_CHECKPOINTING,
}


## Step 2 - Data Contract and Preprocessing

Embedding 튜닝의 핵심은 **학습 데이터의 형태**다. 이 실습에서 사용하는 raw row schema는 아래 네 필드를 중심으로 고정한다.

| 필드 | 역할 | 학습에 미치는 영향 |
|------|------|------------------|
| `query` | 한국어 검색 질문 | 모델이 "검색 의도"를 학습하는 anchor |
| `positive` | 정답 passage | query와 가까워져야 할 target |
| `hard_negative` | 헷갈리지만 오답인 passage | query와 멀어져야 할 대조군. **완전히 무관한 문장이 아니라 헷갈릴 수 있는 문장**이어야 학습 신호가 강하다 |
| `task_instruction` | query 앞에 붙이는 영어 instruction | 모델에게 "이 query가 어떤 검색 작업인지" 힌트를 준다 |

**왜 이 형태인가?**

Contrastive learning은 "정답은 가깝게, 오답은 멀게" 만드는 학습이다.
이를 위해 최소한 (query, positive) 쌍이 필요하고, `hard_negative`가 추가되면 모델이 더 세밀하게 구분하는 법을 배운다. Contrastive Learning의 원리 상세는 Step 5 직전의 '### Contrastive Learning의 원리' 셀에서 다룬다.

**좋은 데이터 vs 나쁜 데이터:**
- 좋은 예: hard_negative가 positive와 같은 주제지만 다른 답을 담고 있음 → 모델이 미세한 차이를 학습
- 나쁜 예: hard_negative가 완전히 다른 주제 → 이미 구분할 수 있어서 학습 신호가 약함

이 노트북은 **retrieval** 품질을 개선하는 실습이다. 즉, "맞는 문서를 잘 찾는가"를 본다. "찾은 문서로 답을 잘 만드는가"는 generation 문제이며 별도 실습(Lab 1 SFT)에서 다룬다.

In [ ]:
# ---------- 데이터 스키마 정의 ----------
# 이 6개 컬럼이 모두 있어야 학습이 가능하다.
REQUIRED_COLUMNS = {
    "query",
    "positive",
    "hard_negative",
    "task_instruction",
    "source_doc",
    "split",
}


def read_jsonl(path: Path) -> pd.DataFrame:
    """JSONL 파일을 한 줄씩 읽어 DataFrame으로 변환한다."""
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


def format_query(
    query: str, task_instruction: str, use_instruction: bool = True
) -> str:
    """query 앞에 영어 instruction prefix를 붙인다.
    Step 4 ablation에서 이 prefix의 효과를 직접 확인한다.
    """
    if use_instruction:
        return f"Instruct: {task_instruction}\nQuery: {query}"
    return query


# ---------- 데이터 로드 ----------
train_df = read_jsonl(TRAIN_PATH)
eval_df = read_jsonl(EVAL_PATH)

# ---------- 데이터 무결성 검증 ----------
# 필수 컬럼 존재 여부 + null 체크
for name, df in [("train", train_df), ("eval", eval_df)]:
    missing = REQUIRED_COLUMNS - set(df.columns)
    assert not missing, f"{name} is missing columns: {missing}"
    assert df["query"].notna().all(), f"{name} has null query"
    assert df["positive"].notna().all(), f"{name} has null positive"
    assert df["hard_negative"].notna().all(), f"{name} has null hard_negative"
    assert df["task_instruction"].notna().all(), f"{name} has null task_instruction"
    # instruction 포함/미포함 두 가지 query 버전을 미리 만들어 둔다
    df["query_with_instruction"] = [
        format_query(q, inst, use_instruction=True)
        for q, inst in zip(df["query"], df["task_instruction"])
    ]
    df["query_without_instruction"] = df["query"]

assert len(eval_df) == NUM_EVAL_QUERIES, (
    f"Expected {NUM_EVAL_QUERIES} hold-out queries, got {len(eval_df)}"
)

# task_instruction이 영어여야 하므로, 한국어가 섞인 행이 있는지 확인한다
instruction_with_korean = (
    train_df["task_instruction"].str.contains(r"[가-힣]", regex=True).sum()
)
print(
    f"Rows whose task_instruction still contains Korean characters: {instruction_with_korean}"
)
print(train_df[["query", "task_instruction", "positive", "hard_negative"]].head(3))

In [ ]:
# ---------- train/eval 오염 검사 ----------
train_queries = set(train_df["query"].str.strip())
eval_queries = set(eval_df["query"].str.strip())
overlap = train_queries & eval_queries
if overlap:
    print(f"⚠ WARNING: {len(overlap)} queries appear in both train and eval sets:")
    for q in list(overlap)[:3]:
        print(f"  - {q[:80]}")
else:
    print(f"✓ No query overlap between train ({len(train_queries)}) and eval ({len(eval_queries)}) sets")

In [ ]:
# ---------- retrieval 평가용 corpus 구축 ----------
# train + eval 모든 행의 positive와 hard_negative를 합쳐서
# 하나의 검색 대상 corpus를 만든다.
# 이렇게 하면 "정답이 아닌 passage도 검색 후보"로 들어가므로
# 모델이 진짜로 구분할 수 있는지를 더 엄밀하게 테스트할 수 있다.

corpus_rows = []
for split_name, df in [("train", train_df), ("eval", eval_df)]:
    for row in df.itertuples(index=False):
        corpus_rows.append(
            {
                "passage": row.positive.strip(),
                "source_doc": row.source_doc,
                "role": "positive",
                "origin_split": split_name,
            }
        )
        corpus_rows.append(
            {
                "passage": row.hard_negative.strip(),
                "source_doc": row.source_doc,
                "role": "hard_negative",
                "origin_split": split_name,
            }
        )

# 중복 passage 제거 후 고유 doc_id 부여
corpus_df = (
    pd.DataFrame(corpus_rows).drop_duplicates(subset=["passage"]).reset_index(drop=True)
)
corpus_df["doc_id"] = [f"doc_{i:03d}" for i in range(len(corpus_df))]

# eval 행의 정답 positive → doc_id 매핑 (hit rate 계산에 사용)
passage_to_doc_id = dict(zip(corpus_df["passage"], corpus_df["doc_id"]))
eval_df["gold_doc_id"] = eval_df["positive"].map(passage_to_doc_id)

assert eval_df["gold_doc_id"].notna().all(), (
    "Some eval positives are missing from the retrieval corpus"
)

print(f"train rows: {len(train_df)}")
print(f"eval rows: {len(eval_df)}")
print(f"retrieval corpus size: {len(corpus_df)}")
corpus_df.head(5)

In [ ]:
# ---------- 토큰 길이 검증 ----------
# MAX_SEQ_LENGTH를 초과하는 텍스트가 있으면 truncation으로 정보 손실 발생
from transformers import AutoTokenizer

_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME.replace("unsloth/", ""))
for col_name, series in [("query_with_instruction", train_df["query_with_instruction"]), 
                          ("positive", train_df["positive"]), 
                          ("hard_negative", train_df["hard_negative"])]:
    lengths = series.apply(lambda x: len(_tokenizer.encode(x)))
    over = (lengths > MAX_SEQ_LENGTH).sum()
    print(f"{col_name}: max={lengths.max()}, mean={lengths.mean():.0f}, over MAX_SEQ_LENGTH={over}")
    if over > 0:
        print(f"  ⚠ {over}건이 MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}을 초과합니다. truncation에 의한 정보 손실 가능.")
del _tokenizer

## Step 3 - Baseline Retrieval Before Tuning

먼저 base embedding으로 hold-out query 30개를 검색한다.
이때 메인 지표는 `Top-3 hit rate`다: 정답 passage가 상위 3개 안에 들어있는 비율이다.

추가로 `MRR@10`(Mean Reciprocal Rank)도 함께 출력한다. 이 값이 높을수록 정답이 더 앞 순위에 위치한다는 뜻이다.

In [ ]:
from unsloth import FastSentenceTransformer


def load_embedding_model(model_name: str):
    """Unsloth의 FastSentenceTransformer로 embedding 모델을 로드한다."""
    return FastSentenceTransformer.from_pretrained(
        model_name=model_name,
        max_seq_length=MAX_SEQ_LENGTH,
        full_finetuning=False,  # LoRA만 학습할 것이므로 False
    )


def encode_texts(model, texts: list[str], batch_size: int = 64) -> np.ndarray:
    """텍스트 리스트를 normalized embedding 벡터로 변환한다.
    normalize_embeddings=True이면 cosine similarity를 dot product로 계산할 수 있다.
    """
    with torch.inference_mode():
        embeddings = model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,  # L2 정규화 → dot product = cosine similarity
        )
    return np.asarray(embeddings, dtype=np.float32)


def short(text: str, limit: int = 90) -> str:
    """표 출력용으로 긴 텍스트를 90자로 자른다."""
    return text if len(text) <= limit else text[:limit] + " ..."


def evaluate_retrieval(model, eval_frame: pd.DataFrame, use_instruction: bool = True):
    """hold-out query들로 retrieval 성능을 측정한다.

    Returns:
        metrics: Top-3 hit rate, Recall@3, MRR@10 딕셔너리
        detail_df: query별 상세 결과 DataFrame
    """
    # 1) query와 corpus를 각각 인코딩
    query_texts = [
        format_query(row.query, row.task_instruction, use_instruction=use_instruction)
        for row in eval_frame.itertuples(index=False)
    ]
    passage_texts = corpus_df["passage"].tolist()

    query_emb = encode_texts(model, query_texts)    # shape: (n_queries, dim)
    passage_emb = encode_texts(model, passage_texts)  # shape: (n_corpus, dim)

    # 2) dot product로 유사도 행렬 계산 (normalized이므로 cosine similarity와 동일)
    scores = np.matmul(query_emb, passage_emb.T)  # shape: (n_queries, n_corpus)

    # 3) 각 query별 상위 TOP_K_MRR개 인덱스 추출
    top_idx = np.argsort(-scores, axis=1)[:, :TOP_K_MRR]

    # 4) query별로 hit@3, mrr@10 등 지표 계산
    rows = []
    for row_number, row in enumerate(eval_frame.itertuples(index=False)):
        ranked = corpus_df.iloc[top_idx[row_number]].copy()
        ranked_scores = scores[row_number, top_idx[row_number]]
        ranked = ranked.assign(score=ranked_scores)
        ranked_doc_ids = ranked["doc_id"].tolist()

        # 정답이 몇 번째에 있는지 찾기 (없으면 None)
        rank = next(
            (
                i + 1
                for i, doc_id in enumerate(ranked_doc_ids)
                if doc_id == row.gold_doc_id
            ),
            None,
        )
        rows.append(
            {
                "query": row.query,
                "source_doc": row.source_doc,
                "gold_doc_id": row.gold_doc_id,
                "hit@3": int(row.gold_doc_id in ranked_doc_ids[:TOP_K_CLASS]),
                "recall@3": int(row.gold_doc_id in ranked_doc_ids[:TOP_K_CLASS]),
                "mrr@10": 0.0 if rank is None else 1.0 / rank,
                "top1_doc_id": ranked.iloc[0]["doc_id"],
                "top1_passage": short(ranked.iloc[0]["passage"]),
                "top3_doc_ids": ranked_doc_ids[:TOP_K_CLASS],
                "top3_passages": [
                    short(x) for x in ranked["passage"].tolist()[:TOP_K_CLASS]
                ],
                "gold_positive_excerpt": short(row.positive),
                "gold_rank": rank,
            }
        )

    detail_df = pd.DataFrame(rows)
    metrics = {
        "top3_hit_rate": detail_df["hit@3"].mean(),
        "recall@3": detail_df["recall@3"].mean(),
        "mrr@10": detail_df["mrr@10"].mean(),
        "n_queries": len(detail_df),
    }
    return metrics, detail_df


# ---------- base 모델 로드 (튜닝 전 상태) ----------
base_model = load_embedding_model(BASE_MODEL_NAME)

In [ ]:
baseline_metrics, baseline_detail = evaluate_retrieval(
    base_model, eval_df, use_instruction=True
)
pd.DataFrame([{"model": "base", **baseline_metrics}]).round(4)

## Step 4 - 실험: Query-side English Instruction Prefix

**실험 질문:** 같은 한국어 query라도, 앞에 영어 instruction을 붙이면 retrieval이 달라질까?

`Qwen3-Embedding`은 **instruction-aware** 모델이다. query 앞에 "이 query는 어떤 검색 작업인가"를 알려주는 영어 instruction을 붙이면, 모델이 query를 해당 작업에 맞게 인코딩한다.

```
# instruction 없이
"가용성 SLA는 어떻게 계산하나요?"

# instruction 포함
"Instruct: Retrieve relevant passages about monitoring and observability\nQuery: 가용성 SLA는 어떻게 계산하나요?"
```

이 instruction은 번역이 아니다. 모델에게 **검색 작업의 종류**를 알려주는 task hint다.

In [ ]:
# ---------- instruction prefix 유무에 따른 retrieval 비교 ----------
# 같은 base 모델, 같은 eval query, instruction만 on/off 전환
ablation_rows = []
for use_instruction in [False, True]:
    metrics, _ = evaluate_retrieval(
        base_model,
        eval_df,
        use_instruction=use_instruction,
    )
    ablation_rows.append(
        {
            "query_side_instruction": "on" if use_instruction else "off",
            **metrics,
        }
    )

# 두 조건의 hit rate를 나란히 비교
ablation_df = pd.DataFrame(ablation_rows)[
    ["query_side_instruction", "top3_hit_rate", "recall@3", "mrr@10", "n_queries"]
]
ablation_df.round(4)

### Ablation 결과 요약

위 실험에서 instruction prefix 유무에 따른 retrieval 차이를 확인했다.

**결론:** instruction prefix가 retrieval 품질에 영향을 주므로, 이후 학습 데이터의 query에도 동일한 instruction prefix를 적용한다. 이것이 `format_query()` 함수가 `Instruct: ... \nQuery: ...` 형식을 만드는 이유다.

## Step 5 - Training: MultipleNegativesRankingLoss (MNRL)

Embedding 튜닝에 사용하는 loss는 `MultipleNegativesRankingLoss`다.

**핵심 원리:** 같은 batch 안에서 각 query의 정답 positive는 가깝게, 다른 query의 positive(= in-batch negative)와 explicit hard_negative는 멀게 만든다.

```mermaid
graph LR
    Q1["query_1"] -->|가깝게 당김| P1["positive_1 ✓"]
    Q1 -.->|밀어냄| P2["positive_2<br/>(in-batch neg)"]
    Q1 -.->|밀어냄| P3["positive_3<br/>(in-batch neg)"]
    Q1 -.->|강하게 밀어냄| N1["negative_1<br/>(explicit hard neg)"]
    
    style P1 fill:#c8e6c9,stroke:#388e3c
    style P2 fill:#ffcdd2,stroke:#d32f2f
    style P3 fill:#ffcdd2,stroke:#d32f2f
    style N1 fill:#ef9a9a,stroke:#c62828
```

- **in-batch negative**: batch 안의 다른 query들의 positive가 자동으로 negative 역할을 한다. batch가 클수록 negative가 많아져 학습 신호가 강해진다.
- **explicit hard negative**: 우리 데이터의 `hard_negative` 컬럼이 이 역할을 한다. 정답과 헷갈리는 passage를 명시적으로 밀어내는 효과가 있다.

수식은 바로 다음 셀 '### MNRL 손실 함수 수식'에서 확인한다.

In [ ]:
# ---------- 학습 데이터셋 구성 ----------
# Sentence Transformers가 기대하는 컬럼명으로 변환:
#   anchor   = instruction이 붙은 query (검색 질문)
#   positive = 정답 passage
#   negative = hard_negative (헷갈리는 오답 passage)
train_tuples = Dataset.from_pandas(
    train_df.assign(
        anchor=train_df.apply(
            lambda row: format_query(
                row["query"], row["task_instruction"], use_instruction=True
            ),
            axis=1,
        ),
        negative=train_df["hard_negative"],
    )[["anchor", "positive", "negative"]],
    preserve_index=False,
)

# 첫 번째 행을 확인하여 anchor/positive/negative 형태가 맞는지 본다
train_tuples[0]

### Step 5-A: 모델 로드 + LoRA 부착

LoRA(Low-Rank Adaptation)는 전체 모델 파라미터를 바꾸지 않고, **작은 어댑터 행렬만 학습**하는 기법이다.

LoRA의 수학적 원리는 Lab 1의 '### LoRA의 수학적 원리' 셀을 참고하라. 핵심: $\Delta W = BA$로 rank-r 행렬 분해를 통해 학습 파라미터를 크게 줄인다.

| 파라미터 | 값 | 의미 |
|---------|-----|------|
| `r` | 32 | LoRA rank. 클수록 표현력↑ 메모리↑. 0.6B 모델에 32는 적절한 중간값 |
| `lora_alpha` | 32 | 스케일링 계수. 보통 r과 같거나 2배로 설정 |
| `target_modules` | q/k/v/o/gate/up/down | attention + FFN 전체에 LoRA 적용 |
| `task_type` | FEATURE_EXTRACTION | embedding 모델이므로 분류가 아닌 특성 추출 |

**왜 Lab 1은 r=16이고 Lab 2는 r=32인가:**
- Lab 1 (4B SFT): 모델이 크므로 각 layer의 hidden dimension이 충분히 넓다. r=16으로 충분한 표현력 확보
- Lab 2 (0.6B embedding): 모델이 작고 hidden dimension이 좁으므로 r=32로 표현력 보상
- `FEATURE_EXTRACTION` task는 전체 벡터 공간의 재배열이 필요하여 표현력 요구가 SFT보다 높다
- 0.6B 모델에서 r=32 vs r=16의 VRAM 차이는 ~100MB 수준으로 미미하다
- 일반 규칙: 작은 모델에는 상대적으로 높은 rank, 큰 모델에는 낮은 rank

In [ ]:
from sentence_transformers import (
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
)
from sentence_transformers.training_args import BatchSamplers
from unsloth import is_bf16_supported

# ---------- base 모델을 다시 로드 (학습용) ----------
model = load_embedding_model(BASE_MODEL_NAME)

# ---------- LoRA 어댑터 부착 ----------
# 전체 모델은 고정하고, 작은 어댑터 행렬만 학습한다.
# 위 마크다운의 파라미터 표를 참고.
model = FastSentenceTransformer.get_peft_model(
    model,
    r=32,                          # LoRA rank: 표현력과 메모리 사이의 중간값
    target_modules=[               # attention + FFN 전체에 LoRA 적용
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=32,                 # 스케일링 계수 (보통 r과 동일)
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
    task_type="FEATURE_EXTRACTION",  # embedding 모델이므로 분류가 아닌 특성 추출
)

### Step 5-B: Loss 함수 생성

위에서 설명한 contrastive 학습을 아래 한 줄이 수행한다. `MultipleNegativesRankingLoss`는 batch 안에서 자동으로 in-batch negative를 만들고, 우리가 넣은 `negative` 컬럼도 explicit hard negative로 활용한다.

In [ ]:
loss = losses.MultipleNegativesRankingLoss(model)

### Step 5-C: Trainer 조립

| 인자 | 값 | 의미 |
|------|-----|------|
| `per_device_train_batch_size` | 32 | batch 내 negative 수에 직접 영향. 클수록 학습 신호↑ 메모리↑ |
| `gradient_accumulation_steps` | 2 | 유효 batch = 32×2 = 64. GPU 메모리를 아끼면서 큰 batch 효과 |
| `learning_rate` | 3e-5 | LoRA 학습에 일반적인 범위 |
| `num_train_epochs` | 1 | 1 epoch로 before/after 차이를 확인. 더 돌리기 전에 데이터 품질부터 |
| `warmup_ratio` | 0.03 | 학습 초기 안정화. 첫 3%는 learning rate를 서서히 올림 |
| `batch_sampler` | NO_DUPLICATES | 같은 batch에 같은 passage가 두 번 들어가지 않도록 보장 |

**batch_size=32의 contrastive learning 영향:**
- In-batch negatives: 각 query당 31개의 implicit negative (같은 batch의 다른 positive)
- Explicit hard negative 포함: 31 + 1 = 32개의 negative per query
- Effective batch = 32 × 2 (grad_accum) = 64 → 63개의 in-batch negatives per update
- batch를 16으로 줄이면 negative가 절반 → 학습 신호가 크게 약해짐 → OOM 시에만 줄일 것
- Sentence Transformers 연구에서 batch_size는 contrastive learning 성능에 가장 큰 영향을 미치는 하이퍼파라미터 중 하나

참고: [Sentence Transformers Training Overview](https://www.sbert.net/docs/sentence_transformer/training_overview.html)

In [ ]:
trainer = SentenceTransformerTrainer(
    model=model,
    train_dataset=train_tuples,
    loss=loss,
    args=SentenceTransformerTrainingArguments(
        output_dir=str(ARTIFACT_DIR / "checkpoints"),
        num_train_epochs=TRAIN_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACC_STEPS,
        learning_rate=LEARNING_RATE,
        fp16=not is_bf16_supported(),
        bf16=is_bf16_supported(),
        logging_steps=10,
        warmup_ratio=0.03,
        lr_scheduler_type="constant_with_warmup",
        save_strategy="epoch",
        report_to="none",
        batch_sampler=BatchSamplers.NO_DUPLICATES,
    ),
)

In [ ]:
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"Reserved memory before training = {start_gpu_memory} GB.")
print(f"Gradient checkpointing = {USE_GRADIENT_CHECKPOINTING}")
print(f"Batch size = {BATCH_SIZE}, grad acc steps = {GRAD_ACC_STEPS}")

## Step 5-1 - Training Run

1 epoch 학습을 실행한다. 데이터 규모에 따라 약 2~5분 소요된다.

학습 중 확인할 것:
- `loss`가 점차 줄어드는지 확인한다. 줄어든다면 모델이 positive와 negative를 구분하는 법을 배우고 있다는 뜻이다.
- 성능을 더 올리고 싶다면 epoch를 늘리기보다 **데이터 품질을 먼저 점검**하는 편이 효과적이다.

In [ ]:
train_result = trainer.train()
model.save_pretrained(str(ARTIFACT_DIR / "qwen3_embedding_lora"))
model.tokenizer.save_pretrained(str(ARTIFACT_DIR / "qwen3_embedding_lora"))

print(train_result.metrics)
print(f"Saved adapter to: {ARTIFACT_DIR / 'qwen3_embedding_lora'}")

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

## Step 5-2 (보너스): 배치 사이즈가 Contrastive Learning에 미치는 영향

> H100 80GB 환경에서 약 15-20분 소요됩니다. 24GB GPU에서는 시간 부족 시 건너뛰세요.

**실험 질문:** 같은 데이터, 같은 모델, 같은 epoch인데 배치 사이즈만 다르면 retrieval 품질이 달라질까?

**가설:** Contrastive learning에서 배치 사이즈는 학습 신호의 강도에 직접 영향을 준다.

| batch_size | in-batch negatives | 의미 |
|-----------|-------------------|------|
| 32 | 31 | 각 query당 31개의 비교 대상 |
| 64 | 63 | 2배 많은 비교 대상 |
| 128 | 127 | 4배 많은 비교 대상 |

배치가 클수록 모델이 "이것과 저것을 어떻게 구분하는가"를 더 풍부하게 학습합니다.
단, **GPU 메모리가 충분할 때만** 큰 배치를 사용할 수 있습니다.

**실험 설계:**
1. 각 배치 사이즈마다 **새 LoRA를 초기화**하고 1 epoch 학습 (공정 비교)
2. 동일한 hold-out query 30개로 Top-3 hit rate 측정
3. 결과를 나란히 비교

In [ ]:
def run_batch_ablation(batch_sizes, train_data, eval_frame):
    """배치 사이즈별로 새 LoRA를 초기화하고 1 epoch 학습 후 retrieval 성능을 비교한다.

    각 배치마다 모델을 새로 로드하므로 공정한 비교가 가능하다.
    GPU 메모리는 매 라운드 후 해제된다.
    """
    ablation_results = []

    for bs in batch_sizes:
        print(f"\n{'='*60}")
        print(f"  Ablation: batch_size={bs} (in-batch negatives: {bs - 1})")
        print(f"{'='*60}")

        # 1. 모델 재로드 + 새 LoRA 부착 (공정 비교를 위해 매번 초기화)
        abl_model = load_embedding_model(BASE_MODEL_NAME)
        abl_model = FastSentenceTransformer.get_peft_model(
            abl_model,
            r=32,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                            "gate_proj", "up_proj", "down_proj"],
            lora_alpha=32,
            lora_dropout=0,
            bias="none",
            use_gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
            random_state=SEED,
            use_rslora=False,
            loftq_config=None,
            task_type="FEATURE_EXTRACTION",
        )

        # 2. Loss + Trainer
        abl_loss = losses.MultipleNegativesRankingLoss(abl_model)
        abl_trainer = SentenceTransformerTrainer(
            model=abl_model,
            train_dataset=train_data,
            loss=abl_loss,
            args=SentenceTransformerTrainingArguments(
                output_dir=str(ARTIFACT_DIR / f"ablation_bs{bs}"),
                num_train_epochs=TRAIN_EPOCHS,
                per_device_train_batch_size=bs,
                gradient_accumulation_steps=1,
                learning_rate=LEARNING_RATE,
                fp16=not is_bf16_supported(),
                bf16=is_bf16_supported(),
                logging_steps=10,
                warmup_ratio=0.03,
                lr_scheduler_type="constant_with_warmup",
                save_strategy="no",
                report_to="none",
                batch_sampler=BatchSamplers.NO_DUPLICATES,
            ),
        )

        # 3. 학습
        train_result = abl_trainer.train()
        train_loss = train_result.metrics.get("train_loss", None)

        # 4. 평가
        metrics, _ = evaluate_retrieval(abl_model, eval_frame, use_instruction=True)

        peak_mem = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
        ablation_results.append({
            "batch_size": bs,
            "in_batch_negatives": bs - 1,
            "top3_hit_rate": metrics["top3_hit_rate"],
            "mrr@10": metrics["mrr@10"],
            "train_loss": round(train_loss, 4) if train_loss else None,
            "peak_gpu_gb": peak_mem,
        })

        print(f"  → Top-3 hit rate: {metrics['top3_hit_rate']:.4f}")
        print(f"  → MRR@10: {metrics['mrr@10']:.4f}")
        print(f"  → Peak GPU: {peak_mem} GB")

        # 5. GPU 메모리 해제
        del abl_model, abl_trainer, abl_loss
        torch.cuda.empty_cache()

    return pd.DataFrame(ablation_results)


print("run_batch_ablation() 정의 완료")

In [ ]:
# ---------- 시각화: 배치 사이즈 vs Retrieval 성능 ----------
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 숫자 배치 사이즈만 필터 (base 제외)
plot_df = ablation_df.copy()

# Top-3 Hit Rate
ax1 = axes[0]
bars1 = ax1.bar(
    [str(bs) for bs in plot_df["batch_size"]],
    plot_df["top3_hit_rate"],
    color=["#ff9999", "#66b3ff", "#99ff99"],
    edgecolor="black",
    alpha=0.8,
)
ax1.axhline(
    y=baseline_metrics["top3_hit_rate"],
    color="gray", linestyle="--", linewidth=1.5,
    label=f"base (no tuning): {baseline_metrics['top3_hit_rate']:.3f}",
)
ax1.set_xlabel("Batch Size")
ax1.set_ylabel("Top-3 Hit Rate")
ax1.set_title("Batch Size vs Top-3 Hit Rate")
ax1.legend()
ax1.grid(axis="y", alpha=0.3)
# 바 위에 in-batch negative 수 주석
for bar, neg in zip(bars1, plot_df["in_batch_negatives"]):
    ax1.annotate(
        f"{neg} neg",
        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
        ha="center", va="bottom", fontsize=9, color="black",
    )

# MRR@10
ax2 = axes[1]
bars2 = ax2.bar(
    [str(bs) for bs in plot_df["batch_size"]],
    plot_df["mrr@10"],
    color=["#ff9999", "#66b3ff", "#99ff99"],
    edgecolor="black",
    alpha=0.8,
)
ax2.axhline(
    y=baseline_metrics["mrr@10"],
    color="gray", linestyle="--", linewidth=1.5,
    label=f"base (no tuning): {baseline_metrics['mrr@10']:.3f}",
)
ax2.set_xlabel("Batch Size")
ax2.set_ylabel("MRR@10")
ax2.set_title("Batch Size vs MRR@10")
ax2.legend()
ax2.grid(axis="y", alpha=0.3)

plt.suptitle("Contrastive Learning: In-Batch Negatives가 Retrieval에 미치는 영향", fontsize=13)
plt.tight_layout()
plt.show()

### Ablation 결과 해석

위 실험에서 확인한 것:

**배치 사이즈가 클수록 retrieval 성능이 향상된다.** 이는 MNRL loss의 근본 원리에서 비롯된다:
- 배치가 크면 각 query당 더 많은 in-batch negative와 비교할 수 있다
- 더 많은 비교 = 더 풍부한 학습 신호 = 더 정밀한 벡터 공간 구성

**하지만 무한정 키울 수는 없다:**
- GPU 메모리 한계: 0.6B 모델에서 bs=256 이상은 대부분의 GPU에서 OOM
- Diminishing returns: 특정 시점 이후로는 추가 negative의 효과가 줄어든다
- 데이터 크기 제약: 배치가 학습 데이터 전체보다 커지면 의미 없음

**실무 권장:**
| GPU VRAM | 권장 배치 | in-batch neg | 비고 |
|---------|----------|-------------|------|
| 80GB (H100) | 128 | 127 | 최적 |
| 40-48GB (A100/L40S) | 64 | 63 | 우수 |
| 24GB (RTX 4090) | 32 | 31 | 기본 |
| 12-16GB (RTX 3060) | 16 | 15 | 최소 |

**핵심 takeaway:** Contrastive learning에서 배치 사이즈는 단순한 학습 속도 파라미터가 아니라 **학습 품질에 직접 영향을 주는 하이퍼파라미터**이다.

---

> 위 ablation이 완료되었으면, 아래 Step 6에서는 **원래 자동 감지된 배치 사이즈로 학습한 결과**를 기준으로 before/after 비교를 진행합니다.

In [ ]:
# ---------- 배치 사이즈 ablation 실행 ----------
# H100 80GB: [32, 64, 128] 모두 실행 가능
# 24GB GPU: [16, 32] 정도로 축소하세요
ABLATION_BATCH_SIZES = [32, 64, 128]

print(f"Ablation 실험: batch_sizes={ABLATION_BATCH_SIZES}")
print(f"예상 소요: {len(ABLATION_BATCH_SIZES)} x ~5분 = ~{len(ABLATION_BATCH_SIZES) * 5}분")
print()

ablation_df = run_batch_ablation(
    batch_sizes=ABLATION_BATCH_SIZES,
    train_data=train_tuples,
    eval_frame=eval_df,
)

# baseline과 함께 비교
ablation_with_baseline = pd.concat([
    pd.DataFrame([{
        "batch_size": "base (no tuning)",
        "in_batch_negatives": "-",
        "top3_hit_rate": baseline_metrics["top3_hit_rate"],
        "mrr@10": baseline_metrics["mrr@10"],
        "train_loss": "-",
        "peak_gpu_gb": "-",
    }]),
    ablation_df,
], ignore_index=True)

ablation_with_baseline.round(4)

## Step 6 - Before/After Retrieval Comparison

이제 숫자와 질적 표를 둘 다 본다.

읽는 순서:
1. `Top-3 hit rate`가 실제로 올랐는지 본다.
2. `MRR@10`으로 정답이 더 앞 순위로 당겨졌는지 본다.
3. 5개 샘플 표에서 top-1이 어떤 문장으로 바뀌었는지 확인한다.
4. 실패 사례도 확인하여 어떤 query가 여전히 어려운지 파악한다.

In [ ]:
# ---------- 튜닝된 모델로 동일 eval query 재평가 ----------
tuned_metrics, tuned_detail = evaluate_retrieval(model, eval_df, use_instruction=True)

# base vs tuned 지표를 나란히 놓고 delta 계산
comparison_df = pd.DataFrame(
    [
        {"model": "base", **baseline_metrics},
        {"model": "tuned", **tuned_metrics},
    ]
)
comparison_df["delta_vs_base_top3"] = (
    comparison_df["top3_hit_rate"] - comparison_df.loc[0, "top3_hit_rate"]
)
comparison_df["delta_vs_base_mrr10"] = (
    comparison_df["mrr@10"] - comparison_df.loc[0, "mrr@10"]
)
comparison_df.round(4)

In [ ]:
# 성공 사례: 추천 쿼리 + 튜닝 후 hit@3 성공 케이스
sample_eval = eval_df[eval_df["query"].isin(RECOMMENDED_SAMPLE_QUERIES)].copy()
if len(sample_eval) < 5:
    sample_eval = pd.concat(
        [
            sample_eval,
            eval_df.loc[~eval_df["query"].isin(sample_eval["query"])].head(
                5 - len(sample_eval)
            ),
        ],
        ignore_index=True,
    )
    sample_eval = sample_eval.drop_duplicates(subset=["query"]).head(5)

baseline_lookup = baseline_detail.set_index("query")
tuned_lookup = tuned_detail.set_index("query")

qual_rows = []
for row in sample_eval.itertuples(index=False):
    base_row = baseline_lookup.loc[row.query]
    tuned_row = tuned_lookup.loc[row.query]
    qual_rows.append(
        {
            "query": row.query,
            "gold_positive_excerpt": short(row.positive),
            "baseline_hit@3": int(base_row["hit@3"]),
            "baseline_top1": base_row["top1_passage"],
            "tuned_hit@3": int(tuned_row["hit@3"]),
            "tuned_top1": tuned_row["top1_passage"],
        }
    )

# 실패 사례: 튜닝 후에도 hit@3 == 0인 query 추가
failed_queries = tuned_detail[tuned_detail["hit@3"] == 0]
if len(failed_queries) > 0:
    for frow in eval_df[eval_df["query"].isin(failed_queries["query"])].head(3).itertuples(index=False):
        if frow.query not in [r["query"] for r in qual_rows]:
            base_row = baseline_lookup.loc[frow.query]
            tuned_row = tuned_lookup.loc[frow.query]
            qual_rows.append(
                {
                    "query": f"[FAIL] {frow.query}",
                    "gold_positive_excerpt": short(frow.positive),
                    "baseline_hit@3": int(base_row["hit@3"]),
                    "baseline_top1": base_row["top1_passage"],
                    "tuned_hit@3": int(tuned_row["hit@3"]),
                    "tuned_top1": tuned_row["top1_passage"],
                }
            )

qualitative_df = pd.DataFrame(qual_rows)
qualitative_df

## Step 7 - 벡터 압축: 품질을 유지하면서 메모리 줄이기

튜닝된 모델이 더 좋은 retrieval을 보여줬다. 이제 실제 배포를 생각해보자.

문서가 10만 개이고 embedding 차원이 1024라면, 정밀도에 따라 메모리 사용량이 크게 달라진다:

| 정밀도 | 차원당 크기 | 1024차원 × 100K 문서 | float32 대비 |
|--------|-----------|---------------------|-------------|
| float32 | 4 bytes | ~391 MB | 1× |
| uint8 | 1 byte | ~98 MB | **4× 절감** |
| binary | 1 bit | ~12 MB | **32× 절감** |

`sentence-transformers`에는 이 압축을 한 줄로 수행하는 `quantize_embeddings()` 함수가 내장되어 있다.

**uint8 양자화 원리:** 각 차원의 최솟값/최댓값 범위를 0~255로 매핑한다. 원래 float32(4바이트)를 uint8(1바이트)로 줄이므로 메모리가 4분의 1이 된다.

**binary 양자화 원리:** 각 차원의 값이 0보다 크면 1, 아니면 0으로 변환한 뒤 8개 bit를 1바이트로 묶는다(bit-packing). 극단적으로 압축되지만 정보 손실이 크다.

In [ ]:
from sentence_transformers.quantization import quantize_embeddings

# 튜닝된 모델로 corpus 임베딩 (float32)
passage_texts = corpus_df["passage"].tolist()
corpus_emb_f32 = encode_texts(model, passage_texts)

# uint8 양자화 (corpus 자체를 calibration으로 사용하여 차원별 min/max 범위 결정)
corpus_emb_uint8 = quantize_embeddings(
    corpus_emb_f32, precision="uint8", calibration_embeddings=corpus_emb_f32
)

# Binary 양자화 (각 차원의 부호만 보존하여 bit-packing)
corpus_emb_binary = quantize_embeddings(corpus_emb_f32, precision="ubinary")

print(f"float32: {corpus_emb_f32.nbytes:,} bytes  (원본)")
print(f"uint8:   {corpus_emb_uint8.nbytes:,} bytes  ({corpus_emb_f32.nbytes / corpus_emb_uint8.nbytes:.0f}× 절감)")
print(f"binary:  {corpus_emb_binary.nbytes:,} bytes  ({corpus_emb_f32.nbytes / corpus_emb_binary.nbytes:.0f}× 절감)")
print(f"\nshape 비교:")
print(f"  float32: {corpus_emb_f32.shape}, dtype={corpus_emb_f32.dtype}")
print(f"  uint8:   {corpus_emb_uint8.shape}, dtype={corpus_emb_uint8.dtype}")
print(f"  binary:  {corpus_emb_binary.shape}, dtype={corpus_emb_binary.dtype}")

### 양자화된 벡터로 검색하면 품질이 얼마나 떨어질까?

`semantic_search_faiss`는 2단계 검색을 지원한다:

1. **1단계**: 양자화된 corpus에서 `top_k * rescore_multiplier`개 후보를 빠르게 뽑는다.
2. **2단계 (rescore)**: float32 query embedding으로 후보들을 재점수 매겨 최종 top_k를 선정한다.

이렇게 하면 **corpus는 압축된 상태로 메모리를 절약**하면서, **최종 순위는 float32 수준의 정확도**를 유지할 수 있다.

In [ ]:
from sentence_transformers.quantization import semantic_search_faiss

# eval query 인코딩 (float32 유지 — rescore에 사용)
eval_query_texts = [
    format_query(row.query, row.task_instruction, use_instruction=True)
    for row in eval_df.itertuples(index=False)
]
query_emb_f32 = encode_texts(model, eval_query_texts)

# gold_doc_id -> corpus index 매핑
doc_id_to_idx = {doc_id: idx for idx, doc_id in enumerate(corpus_df["doc_id"])}

quant_results = {}
for precision, corpus_q in [("uint8", corpus_emb_uint8), ("ubinary", corpus_emb_binary)]:
    results, search_time, _ = semantic_search_faiss(
        query_embeddings=query_emb_f32,
        corpus_embeddings=corpus_q,
        corpus_precision=precision,
        top_k=TOP_K_MRR,
        calibration_embeddings=corpus_emb_f32 if precision == "uint8" else None,
        rescore=True,
        rescore_multiplier=4,
        output_index=False,
    )
    # Top-3 hit rate 계산
    hits = 0
    for i, row in enumerate(eval_df.itertuples(index=False)):
        gold_idx = doc_id_to_idx[row.gold_doc_id]
        top_k_ids = [r["corpus_id"] for r in results[i][:TOP_K_CLASS]]
        if gold_idx in top_k_ids:
            hits += 1
    hit_rate = hits / len(eval_df)
    quant_results[precision] = {
        "top3_hit_rate": hit_rate,
        "search_time_ms": round(search_time * 1000, 1),
        "memory_bytes": corpus_q.nbytes,
    }
    print(f"{precision}: Top-3 hit rate = {hit_rate:.4f}, search time = {search_time*1000:.1f}ms")

In [ ]:
# 전체 비교: base -> tuned -> uint8 -> binary
final_comparison = pd.DataFrame([
    {
        "model": "base (float32)",
        "top3_hit_rate": baseline_metrics["top3_hit_rate"],
        "precision": "float32",
        "corpus_memory_bytes": corpus_emb_f32.nbytes,
    },
    {
        "model": "tuned (float32)",
        "top3_hit_rate": tuned_metrics["top3_hit_rate"],
        "precision": "float32",
        "corpus_memory_bytes": corpus_emb_f32.nbytes,
    },
    {
        "model": "tuned (uint8 + rescore)",
        "top3_hit_rate": quant_results["uint8"]["top3_hit_rate"],
        "precision": "uint8",
        "corpus_memory_bytes": quant_results["uint8"]["memory_bytes"],
    },
    {
        "model": "tuned (binary + rescore)",
        "top3_hit_rate": quant_results["ubinary"]["top3_hit_rate"],
        "precision": "ubinary",
        "corpus_memory_bytes": quant_results["ubinary"]["memory_bytes"],
    },
])
final_comparison["memory_reduction"] = (
    final_comparison["corpus_memory_bytes"].iloc[0] / final_comparison["corpus_memory_bytes"]
).apply(lambda x: f"{x:.0f}x")
final_comparison.round(4)

### 벡터 압축 해석

- **uint8 (4x 절감)**: rescore 덕분에 hit rate가 float32와 거의 동일하다. 대부분의 배포 시나리오에서 최적 선택.
- **binary (32x 절감)**: hit rate가 소폭 하락할 수 있다. 극단적 메모리 제약이거나 1차 후보 필터링용으로 적합.
- **rescore=True가 핵심**: float32 query로 재점수를 매기므로 양자화에 의한 순위 손실을 보상한다.

실무 판단 기준:

| corpus 규모 | 권장 정밀도 | 이유 |
|------------|-----------|------|
| 100K 이하 | float32 | 수백 MB 수준이므로 압축 불필요 |
| 1M+ | uint8 + rescore | 메모리 4x 절감, 품질 거의 유지 |
| 10M+ | binary -> float32 rerank | 1차 필터를 binary로, 최종 순위를 float32로 |

## Step 8 - 결과 해석

### 해석 체크리스트

- **Top-3 hit rate가 올랐는가?** → embedding tuning이 retrieval을 개선한 것이다.
- **MRR@10이 올랐는가?** → 정답이 더 앞 순위로 이동한 것이다.
- **실패한 query가 있는가?** → 해당 query의 hard_negative와 positive를 다시 살펴보자. 데이터 품질 문제일 수 있다.
- **정답을 찾았는데 QA 답변이 이상하다면?** → 그건 generation/prompting 문제다 (이 노트북의 범위 밖).

### 핵심 구분

| 문제 유형 | 증상 | 해결 방향 |
|----------|------|----------|
| **retrieval 문제** | 맞는 문서를 못 찾음 | embedding tuning (이 노트북) |
| **generation 문제** | 맞는 문서를 찾았는데 답이 이상함 | prompting 또는 SFT (Lab 1) |

### Lab 1 + Lab 2 = RAG Pipeline

두 실습의 결과물을 합치면 완전한 도메인 특화 RAG 파이프라인이 됩니다:

| 단계 | 모델 | 역할 |
|------|------|------|
| 1. 사용자 query 입력 | - | 검색 질문 |
| 2. Query → Vector | **Lab 2** tuned embedding | query를 도메인 최적화된 벡터로 변환 |
| 3. Vector Search | FAISS / vector DB | corpus에서 가장 유사한 passage top-k 검색 |
| 4. Context → Response | **Lab 1** tuned SFT model | 검색된 passage를 context로 답변 생성 |

**왜 같은 도메인 데이터로 튜닝하는가:**
- Lab 2의 embedding이 "AgentOps 용어"를 이해하므로 검색 정확도 향상
- Lab 1의 SFT model이 "AgentOps 문맥"에서 답변하므로 생성 품질 향상
- 두 모델의 도메인 이해가 일관되어 end-to-end 성능이 개선됨

**배포 시 고려사항:**
- Embedding model (0.6B): 가볍지만 항상 활성화 (모든 query에 필요)
- SFT model (4B): 무겁지만 요청 시에만 활성화
- → Day-07에서 vLLM으로 이 모델들을 효율적으로 서빙하는 방법을 배웁니다

## Exercise

1. `task_instruction`을 더 구체적인 영어 문장으로 바꿔 본다.
2. 같은 query에 대해 instruction을 제거했을 때와 비교한다.
3. retrieval이 좋아졌는지, 아니면 단순히 generation 설명 문장이 좋아 보이는 착시인지 구분해서 적는다.
4. 위 Step 7에서 `rescore=False`로 바꿔보고, uint8 검색의 hit rate 변화를 확인한다.
5. `calibration_embeddings=None`으로 양자화한 뒤 검색 결과가 어떻게 달라지는지 비교한다.

In [ ]:
# Exercise answer scaffold
experiment_query = eval_df.iloc[0]["query"]
experiment_instructions = [
    eval_df.iloc[0]["task_instruction"],
    "Retrieve the passage that best answers a monitoring and observability question.",
    "",
]

for instruction in experiment_instructions:
    use_instruction = bool(instruction)
    rendered_query = format_query(
        experiment_query, instruction, use_instruction=use_instruction
    )
    print("=" * 100)
    print(rendered_query)